# Part 1: Training Notebook - Custom CIFAR-10 Adversarial Training

This notebook focuses on training the **Custom ResNetV2-BiT** model, as defined in your project notebook. It supports standard and adversarial training.

In [ ]:
!pip install timm torchinfo

## Custom Model Definition
This matches the `Model` class from your original notebook, optimized for CIFAR-10.

In [ ]:
import torch
import torch.nn as nn
import timm

class Model(nn.Module):
    def __init__(self, model_name='resnetv2_50x1_bit', num_classes=10):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=True)
        self.model.stem.conv = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.model.stem.pool = nn.Identity()

        for param in self.model.parameters():
            param.requires_grad = False
        for param in self.model.stem.conv.parameters():
            param.requires_grad = True
        for param in self.model.stages[3].parameters():
            param.requires_grad = True

        in_channels = self.model.head.fc.in_channels 
        self.model.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(in_channels, num_classes)
        )
        for param in self.model.head.parameters():
            param.requires_grad = True
    def forward(self, x):
        return self.model(x)

In [ ]:
%%writefile attacks.py
import torch
import torch.nn.functional as F

class AdversarialAttack:
    def __init__(self, model, epsilon=0.1, alpha=0.01, iters=10):
        self.model = model
        self.epsilon = epsilon
        self.alpha = alpha
        self.iters = iters
    def fgsm(self, images, labels):
        images = images.clone().detach().to(images.device)
        labels = labels.clone().detach().to(images.device)
        images.requires_grad = True
        outputs = self.model(images)
        loss = F.cross_entropy(outputs, labels)
        self.model.zero_grad()
        loss.backward()
        perturbed_image = images + self.epsilon * images.grad.data.sign()
        return torch.clamp(perturbed_image, -3, 3)
    def pgd(self, images, labels):
        ori_images = images.clone().detach().to(images.device)
        images = images.clone().detach().to(images.device)
        for i in range(self.iters):
            images.requires_grad = True
            outputs = self.model(images)
            loss = F.cross_entropy(outputs, labels)
            self.model.zero_grad()
            loss.backward()
            adv_images = images + self.alpha * images.grad.data.sign()
            eta = torch.clamp(adv_images - ori_images, min=-self.epsilon, max=self.epsilon)
            images = (ori_images + eta).detach()
        return images

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from attacks import AdversarialAttack

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
train_loader = DataLoader(datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train), batch_size=128, shuffle=True)

model = Model(num_classes=10).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
attacker = AdversarialAttack(model, epsilon=8/255.0, alpha=2/255.0, iters=10)

def train(model, loader, adv=False):
    model.train()
    for i, (data, target) in enumerate(loader):
        data, target = data.to(DEVICE), target.to(DEVICE)
        if adv:
            with torch.enable_grad():
                data = attacker.pgd(data, target)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if i % 100 == 0: print(f"Batch {i} done.")

train(model, train_loader, adv=True)
torch.save(model.state_dict(), "resnet_bit_cifar10.pth")